In [2]:
# ============================================================
# 06_descriptive_stats.ipynb
# Descriptive Statistics & Selection Bias Check
#
# Outputs:
#   Table 1. Descriptive statistics by age group
#   Table S1. Completeness analysis (selection bias)
# ============================================================


# ─────────────────────────────────────────────
# Cell 1 | Imports & Paths
# ─────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')

BASE_OUT = os.path.join('..', 'outputs')

DISEASE_CONFIG = {
    'htn': {
        'label'       : 'Hypertension',
        'target'      : 'HE_HP',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_NA', 'N_K'],
    },
    'dm': {
        'label'       : 'Diabetes',
        'target'      : 'HE_DM_HbA1c',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_SUGAR', 'N_CHO', 'N_EN'],
    },
    'dys': {
        'label'       : 'Dyslipidemia',
        'target'      : 'HE_HCHOL',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_FAT', 'N_CHOL'],
    },
}

# Continuous variables to summarise as mean ± SD
CONT_VARS = {
    'age'    : 'Age (years)',
    'HE_BMI' : 'BMI (kg/m²)',
    'HE_wc'  : 'Waist circumference (cm)',
    'HE_wt'  : 'Body weight (kg)',
}

# Categorical variables to summarise as N (%)
CAT_VARS = {
    'BS3_1'    : ('Current smoking', 1),
    'pa_aerobic': ('Aerobic PA practice', 1),
    'BH1'      : ('Health checkup (past year)', 1),
}

# Disease-specific continuous additions
DISEASE_CONT = {
    'htn': {'N_NA': 'Sodium intake (mg)',
             'N_K' : 'Potassium intake (mg)'},
    'dm' : {'N_SUGAR': 'Sugar intake (g)',
             'N_CHO'  : 'Carbohydrate intake (g)',
             'N_EN'   : 'Energy intake (kcal)'},
    'dys': {'N_FAT'  : 'Fat intake (g)',
             'N_CHOL' : 'Dietary cholesterol (mg)'},
}

AGE_BINS   = [18, 44, 64, 120]
AGE_LABELS = ['Young (19–44)', 'Middle (45–64)',
               'Elderly (65+)']

print("Configuration loaded.")


# ─────────────────────────────────────────────
# Cell 2 | Descriptive Statistics Function
# ─────────────────────────────────────────────
def describe_group(df, cont_vars, cat_vars,
                   target_col):
    """
    Compute mean±SD for continuous variables and
    N(%) for categorical variables.
    Returns a dict of {label: 'value string'}.
    """
    result = {'N': str(len(df))}

    for var, label in cont_vars.items():
        if var not in df.columns:
            result[label] = 'N/A'
            continue
        m = df[var].mean()
        s = df[var].std()
        result[label] = f"{m:.1f} ± {s:.1f}"

    # Target prevalence
    if target_col in df.columns:
        prev = df[target_col].mean() * 100
        result['Prevalence (%)'] = f"{prev:.1f}%"

    # Sex
    if 'sex' in df.columns:
        female_pct = (df['sex'] == 2).mean() * 100
        result['Female (%)'] = f"{female_pct:.1f}%"

    for var, (label, positive_val) in cat_vars.items():
        if var not in df.columns:
            result[label] = 'N/A'
            continue
        n   = (df[var] == positive_val).sum()
        pct = n / len(df) * 100
        result[label] = f"{n:,} ({pct:.1f}%)"

    return result


def kruskal_test(df, var, age_col='age_group'):
    """Kruskal-Wallis test across age groups."""
    groups = [
        df[df[age_col] == ag][var].dropna().values
        for ag in ['young', 'middle', 'elderly']
        if ag in df[age_col].unique()
    ]
    if len(groups) < 2:
        return np.nan, np.nan
    stat, p = stats.kruskal(*groups)
    return round(stat, 3), round(p, 4)


# ─────────────────────────────────────────────
# Cell 3 | Generate Table 1 per Disease
# ─────────────────────────────────────────────
for d_key, d_info in DISEASE_CONFIG.items():
    label      = d_info['label']
    target_col = d_info['target']

    df = pd.read_csv(
        os.path.join(BASE_OUT, f"{d_key}_total.csv")
    )
    df['age_group'] = pd.cut(
        df['age'],
        bins=AGE_BINS,
        labels=['young', 'middle', 'elderly'],
        right=True
    )

    # Merge continuous vars
    cont_vars_all = dict(CONT_VARS)
    cont_vars_all.update(DISEASE_CONT[d_key])

    # Groups
    groups = {
        'Total'         : df,
        'Young (19–44)' : df[df['age_group'] == 'young'],
        'Middle (45–64)': df[df['age_group'] == 'middle'],
        'Elderly (65+)' : df[df['age_group'] == 'elderly'],
    }

    rows = []
    for grp_label, grp_df in groups.items():
        desc = describe_group(
            grp_df, cont_vars_all,
            CAT_VARS, target_col
        )
        desc['Group'] = grp_label
        rows.append(desc)

    df_desc = pd.DataFrame(rows).set_index('Group')

    # Add p-values (Kruskal-Wallis, total vs age groups)
    p_row = {'Group': 'p-value (K-W)'}
    for var, lbl in cont_vars_all.items():
        if var in df.columns:
            _, p = kruskal_test(df, var)
            p_str = f"<0.001" if p < 0.001 \
                    else f"{p:.3f}"
            p_row[lbl] = p_str
    p_row['N'] = ''
    p_row['Prevalence (%)'] = ''
    p_row['Female (%)'] = ''
    for _, (lbl, _) in CAT_VARS.items():
        p_row[lbl] = ''

    df_p = pd.DataFrame([p_row]).set_index('Group')
    df_final = pd.concat([df_desc, df_p])

    print(f"\n{'='*70}")
    print(f"  Table 1. Descriptive Statistics — {label}")
    print(f"{'='*70}")
    print(df_final.T.to_string())

    # Save
    path = os.path.join(
        BASE_OUT, f"table1_descriptive_{d_key}.csv"
    )
    df_final.T.to_csv(path)
    print(f"\n  Saved : {path}")


# ─────────────────────────────────────────────
# Cell 4 | Selection Bias Check (Table S1)
# ─────────────────────────────────────────────
# Compare retained vs excluded samples to check
# whether complete-case analysis introduced bias.
# Uses the raw merged data reconstructed from
# per-disease CSVs (proxy: compare age/sex/BMI
# distributions across diseases).

print(f"\n{'='*70}")
print("  Table S1. Selection Bias Check")
print("  (Retained sample characteristics)")
print(f"{'='*70}")

bias_rows = []
for d_key, d_info in DISEASE_CONFIG.items():
    df = pd.read_csv(
        os.path.join(BASE_OUT, f"{d_key}_total.csv")
    )
    bias_rows.append({
        'Disease'         : d_info['label'],
        'N_retained'      : len(df),
        'Age_mean'        : round(df['age'].mean(), 1),
        'Age_sd'          : round(df['age'].std(), 1),
        'Female_pct'      : round(
            (df['sex'] == 2).mean() * 100, 1),
        'BMI_mean'        : round(df['HE_BMI'].mean(), 2),
        'BMI_sd'          : round(df['HE_BMI'].std(), 2),
        'Prevalence_pct'  : round(
            df[d_info['target']].mean() * 100, 1),
    })

df_bias = pd.DataFrame(bias_rows)
print(df_bias.to_string(index=False))

bias_path = os.path.join(BASE_OUT,
                          'table_s1_selection_bias.csv')
df_bias.to_csv(bias_path, index=False)
print(f"\nTable S1 saved : {bias_path}")
print("\n=== 06_descriptive_stats.ipynb complete ===")

Configuration loaded.

  Table 1. Descriptive Statistics — Hypertension
Group                                 Total    Young (19–44)   Middle (45–64)    Elderly (65+) p-value (K-W)
N                                     14361             5813             5434             3114              
Age (years)                     49.6 ± 16.6       32.8 ± 7.5       54.5 ± 5.7       72.3 ± 5.0        <0.001
BMI (kg/m²)                      23.4 ± 3.3       23.0 ± 3.6       23.6 ± 3.2       23.8 ± 3.1        <0.001
Waist circumference (cm)        81.4 ± 10.2      78.6 ± 10.4       82.1 ± 9.5       85.7 ± 9.2        <0.001
Body weight (kg)                62.8 ± 11.7      64.2 ± 12.8      62.9 ± 11.1       60.2 ± 9.9        <0.001
Sodium intake (mg)          2998.2 ± 1556.9  3028.2 ± 1550.8  3071.5 ± 1555.8  2814.4 ± 1556.4        <0.001
Potassium intake (mg)       2595.1 ± 1128.5  2381.3 ± 1045.2  2793.0 ± 1129.3  2649.0 ± 1205.1        <0.001
Prevalence (%)                        32.2%            1